<a href="https://colab.research.google.com/github/vikrant48/AI-ML-All-AI-related-python-codes-/blob/main/ADV_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install langchain langchain-community langchain-huggingface langchain-text-splitters langchain-core langchain-groq
!pip install chromadb sentence-transformers pypdf rank-bm25 ragas datasets

In [ ]:
import os
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
os.environ["LANGCHAIN_API_KEY"] = userdata.get("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"]= userdata.get("LANGCHAIN_TRACING_V2")

In [ ]:
# for embedding
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en"   # better for retrieval than MiniLM
)

In [ ]:
#for reanking
from sentence_transformers import CrossEncoder

reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

In [ ]:
#for LLM
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.2
)

In [ ]:
import re

def extract_name(text):
    lines = text.split("\n")

    for line in lines[:5]:  # check first 5 lines
        line = line.strip()

        # simple rule: name = line with only alphabets and spaces
        if re.match(r"^[A-Za-z ]{3,}$", line):
            return line

    return "Unknown"

In [ ]:
# create chunks

from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=100
)

In [ ]:
# here for multiple resume (load pdf and chunking )
from langchain_community.document_loaders import PyPDFLoader

resume_files = [
    "resume1.pdf",
    "resume2.pdf",
    "resume3.pdf"
]
# for store chunking
chunks = []

for file in resume_files:
    loader = PyPDFLoader(file)
    docs = loader.load()

    if not docs:
        print(f"Skipping empty file: {file}")
        continue

    first_page_text = docs[0].page_content
    name = extract_name(first_page_text)
    print("Extracted Name:", name)

    for doc in docs:
        doc.metadata["name"] = name
        doc.metadata["source"] = file   # useful later

    file_chunk = splitter.split_documents(docs)

    chunks.extend(file_chunk)



create corpus

In [ ]:
corpus_texts = [chunk.page_content for chunk in chunks]
corpus_tokens = [text.lower().split() for text in corpus_texts]

In [ ]:
#Create BM25
from rank_bm25 import BM25Okapi

bm25 = BM25Okapi(corpus_tokens)

In [ ]:
#Embedding + storing
from langchain_community.vectorstores import Chroma

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db"
)

In [ ]:
query = """
Looking for a Java backend engineer with 3+ years experience,
Spring Boot, AWS, microservices, REST APIs, and database knowledge
"""

here retiver method changes


```
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 2, "fetch_k": 10}
)

docs = retriever.invoke(query)
```

to
docs = hybrid_search(query, k=5)




In [ ]:
# map vector results → chunk indices
def get_dense_rankings(query, k):
    dense_docs = vectorstore.similarity_search(query, k=k)

    ranked_indices = []
    for doc in dense_docs:
        for i, chunk in enumerate(chunks):
            if chunk.page_content == doc.page_content:
                ranked_indices.append(i)
                break

    return ranked_indices

In [ ]:
# BM25 rankings
import numpy as np

def get_bm25_rankings(query, k):
    scores = bm25.get_scores(query.lower().split())
    ranked_indices = np.argsort(scores)[::-1][:k]
    return ranked_indices.tolist()

In [ ]:
# with RRF
def reciprocal_rank_fusion(result_lists, k=60):
    scores = {}

    for results in result_lists:
        for rank, idx in enumerate(results):
            scores[idx] = scores.get(idx, 0) + 1 / (k + rank + 1)

    return sorted(scores.items(), key=lambda x: x[1], reverse=True)

In [ ]:
# this is for ranked
def rerank_results(query, results, top_n=5):
    docs = [doc for doc, _ in results]

    pairs = [(query, doc.page_content) for doc in docs]

    scores = reranker.predict(pairs)

    ranked = sorted(zip(scores, docs), key=lambda x: x[0], reverse=True)

    return ranked[:top_n]

In [ ]:
import numpy as np

def hybrid_search(query, k=5):
    # Dense search (semantic)
    dense_results = vectorstore.similarity_search(query, k=k)

    # BM25 search (keyword)
    bm25_scores = bm25.get_scores(query.lower().split())
    top_bm25_idx = np.argsort(bm25_scores)[::-1][:k]

    # Merge (simple union instead of RRF for now)
    results = []

    # Add dense
    for doc in dense_results:
        results.append(doc)

    # Add BM25
    for idx in top_bm25_idx:
        results.append(chunks[idx])

    return results[:k]

In [ ]:
def hybrid_search(query, k=5):
    dense_rank = get_dense_rankings(query, k=10)
    bm25_rank = get_bm25_rankings(query, k=10)

    fused = reciprocal_rank_fusion([dense_rank, bm25_rank])

    results = []
    seen = set()

    for idx, score in fused:
        if idx not in seen:
            results.append((chunks[idx], score))
            seen.add(idx)

        if len(results) >= k:
            break

    return results

maunal context  creation

In [ ]:
docs = hybrid_search(query, k=5)

for doc, score in docs:
    print("Score:", round(score, 4))
    print("Source:", doc.metadata.get("source"))
    print(doc.page_content[:200])
    print("-" * 50)

In [ ]:
reranked = rerank_results(query, docs, top_n=5)
print("Reranked Results:")
for score, doc in reranked:
    print("Score:", round(score, 4))
    print("Source:", doc.metadata.get("source"))
    print(doc.page_content)

In [ ]:
context_parts = []

for i, (doc, score) in enumerate(docs):
    meta = doc.metadata

    context_parts.append(f"""
Candidate {i+1}
Name: {meta.get('name')}
Experience: {meta.get('years')} years
Source: {meta.get('source')}
Match Score: {score:.4f}

Resume:
{doc.page_content}
""")

context = "\n\n----------------\n\n".join(context_parts)
print(context)

In [ ]:
context_parts = []

for i, (score, doc) in enumerate(reranked):
    meta = doc.metadata

    context_parts.append(f"""
Candidate {i+1}
Name: {meta.get('name')}
Experience: {meta.get('years')} years
Source: {meta.get('source')}
Relevance Score: {round(float(score), 4)}

Resume:
{doc.page_content[:10000]}
""")

context = "\n\n----------------------\n\n".join(context_parts)
print(context)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

chat_history = []

prompt = ChatPromptTemplate.from_template("""
Answer the question based only on the context below:

Chat History:
{chat_history}

Context:
{context}

Question:
{question}
""")

In [ ]:
def format_chat_history(chat_history):
    print("\n--- CHAT HISTORY ---\n", chat_history)
    return "\n".join([
        f"User: {q}\nAssistant: {a}" for q, a in chat_history
    ])

In [ ]:
def format_docs(docs):
    return "\n\n".join([doc.page_content for doc in docs])

In [ ]:
# context by method
def get_context(query):
    results = hybrid_search(query, k=15)   # [(doc, score)]
    docs = [doc for doc, _ in results]
    return format_docs(docs)

In [ ]:

chain = (
    {
        "context": get_context,
        "question": RunnablePassthrough(),
        "chat_history": lambda x: format_chat_history(chat_history)
    }
    | prompt
    | llm
    | StrOutputParser()
)

print(chain)

In [ ]:
ques = "give me all candidate name and their skills "
ans = chain.invoke(ques)

print(ans)

chat_history.append((ques, ans))

RASAS (by default it support for open AI but we are using groq )

In [ ]:
questions = [
    "Find Java backend engineer with AWS",
    "Who has experience in Spring Boot?",
    "Give candidates with microservices experience"
]

In [ ]:
from datasets import Dataset

eval_data = {
    "question": [],
    "answer": [],
    "contexts": [],
    "ground_truth": []
}

for q in questions:
    # 1. Get context (your hybrid search)
    results = hybrid_search(q, k=10)

    docs = [doc for doc, _ in results]
    context_texts = [doc.page_content for doc in docs]

    # 2. Get answer (LLM)
    ans = chain.invoke(q)

    # 3. Append
    eval_data["question"].append(q)
    eval_data["answer"].append(ans)
    eval_data["contexts"].append(context_texts)

    # ⚠️ For now (no real GT)
    eval_data["ground_truth"].append(ans)

dataset = Dataset.from_dict(eval_data)

In [ ]:
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall
)

results = evaluate(
    dataset,
    metrics=[
        faithfulness,
        answer_relevancy,
        context_precision,
        context_recall
    ]
)

print(results)